In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data import get_username_mappings, load_backtest_results

USERNAME_TO_PLAYER, _ = get_username_mappings()

MODELS   = ['decay', 'gradient_boost', 'random_forest', 'logistic']
N_VALUES = [1, 3, 8, 10]
COLORS   = {'decay': 'steelblue', 'gradient_boost': 'salmon',
             'random_forest': 'mediumseagreen', 'logistic': 'orchid'}

YES_COLS = [f'p_top{k}_yes_performance' for k in N_VALUES]
NO_COLS  = [f'p_top{k}_no_performance'  for k in N_VALUES]
ROI_COLS = YES_COLS + NO_COLS

def load_and_score(model):
    df = load_backtest_results(model)
    df = df[df['date']>'2025-11-11'].dropna()
    for k in N_VALUES:
        p  = df[f'p_top{k}']
        y  = df[f'in_top{k}']
        mv = k / 100
        df[f'p_top{k}_yes_performance'] = np.where(p > mv,  y       / p,       np.nan)
        df[f'p_top{k}_no_performance']  = np.where(p > mv, (1 - y) / (1 - p), np.nan)
    return df

dfs = {m: load_and_score(m) for m in MODELS}
print({m: len(dfs[m]) for m in MODELS}, 'total pool rows per model')
print({m: dfs[m]['participated'].mean() for m in MODELS}, 'participation rate')

## 1. Overall ROI Summary
Mean return per \$1 bet across all player-tournament pairs where `p_top{k} > k/100`.  
**1.0 = breakeven**, >1.0 = profitable. No-shows count as YES losses / NO wins.

In [55]:
def roi_summary(level='overall'):
    rows = {}
    for m in MODELS:
        if level == 'overall':
            rows[m] = dfs[m][ROI_COLS].mean()
        elif level == 'by_tournament':
            rows[m] = dfs[m].groupby('date')[ROI_COLS].mean().mean()
        elif level == 'by_player':
            rows[m] = dfs[m].groupby('username')[ROI_COLS].mean().mean()
    return pd.DataFrame(rows).T.round(4)

print('=== Overall (equal weight per player-tournament bet) ===')
display(roi_summary('overall'))

print('\n=== Tournament-averaged (equal weight per tournament) ===')
display(roi_summary('by_tournament'))

print('\n=== Player-averaged (equal weight per player) ===')
display(roi_summary('by_player'))

=== Overall (equal weight per player-tournament bet) ===


,p_top1_yes_performance,p_top3_yes_performance,p_top8_yes_performance,p_top10_yes_performance,p_top1_no_performance,p_top3_no_performance,p_top8_no_performance,p_top10_no_performance
decay,0.8804,1.0685,1.0354,1.0295,1.0054,0.9958,1.0029,1.0058
gradient_boost,0.9966,1.0526,0.9829,0.9920,1.0056,0.9965,1.0106,1.0135
random_forest,0.9774,1.0805,0.9691,0.9561,1.0056,0.9956,1.0122,1.0165
logistic,1.1843,1.0525,0.9855,1.0111,1.0030,0.9954,1.0074,1.0060



=== Tournament-averaged (equal weight per tournament) ===


,p_top1_yes_performance,p_top3_yes_performance,p_top8_yes_performance,p_top10_yes_performance,p_top1_no_performance,p_top3_no_performance,p_top8_no_performance,p_top10_no_performance
decay,0.8473,1.0698,1.0371,1.0288,1.0061,0.9961,1.0022,1.0062
gradient_boost,0.9803,1.0535,0.9843,0.9929,1.0061,0.9966,1.0112,1.0139
random_forest,0.9659,1.0691,0.9671,0.9556,1.0061,0.9960,1.0129,1.0166
logistic,1.1783,1.0537,0.9828,1.0095,1.0033,0.9954,1.0075,1.0068



=== Player-averaged (equal weight per player) ===


,p_top1_yes_performance,p_top3_yes_performance,p_top8_yes_performance,p_top10_yes_performance,p_top1_no_performance,p_top3_no_performance,p_top8_no_performance,p_top10_no_performance
decay,0.6870,0.8340,0.8116,0.7836,1.0070,1.0061,1.0245,1.0351
gradient_boost,0.7061,0.8566,0.8024,0.7482,1.0079,1.0069,1.0282,1.0439
random_forest,0.7113,0.8146,0.7858,0.7511,1.0081,1.0077,1.0305,1.0419
logistic,0.8841,0.7462,0.8294,0.7618,1.0058,1.0093,1.0235,1.0384


## 2. ROI by Tournament

In [56]:
by_tourn = pd.concat(
    {m: dfs[m].groupby('date')[ROI_COLS].mean().round(4) for m in MODELS},
    names=['model', 'date']
).sort_index()

by_tourn

p_top1_yes_performance  p_top3_yes_performance  \
model         date                                                         
decay         2025-11-18                  0.4303                  0.8319   
              2025-11-25                  0.0000                  1.7890   
              2025-12-02                  0.1969                  1.1018   
              2025-12-09                  0.0000                  1.1664   
              2025-12-16                  0.2043                  1.5464   
...                                          ...                     ...   
random_forest 2026-06-09                  1.0929                  1.6013   
              2026-06-16                  1.5635                  0.5254   
              2026-06-23                  0.0000                  0.2995   
              2026-06-30                  3.0713                  2.7345   
              2026-07-07                  3.5063                  1.3991   

                          p_top8_yes_performance  p_top10_yes_performance  \
model         date                                                          
decay         2025-11-18                  1.3181                   1.2007   
              2025-11-25                  1.1835                   0.9370   
              2025-12-02                  0.4633                   0.3677   
              2025-12-09                  1.5439                   1.3465   
              2025-12-16                  1.4146                   1.4001   
...                                          ...                      ...   
random_forest 2026-06-09                  1.0933                   0.9950   
              2026-06-16                  0.9301                   0.7785   
              2026-06-23                  0.6009                   0.7055   
              2026-06-30                  1.2271                   1.3381   
              2026-07-07                  0.9646                   0.8089   

                          p_top1_no_performance  p_top3_no_performance  \
model         date                                                       
decay         2025-11-18                 0.9883                 1.0102   
              2025-11-25                 1.0513                 0.9560   
              2025-12-02                 0.9843                 0.9493   
              2025-12-09                 1.0438                 1.0268   
              2025-12-16                 0.9870                 0.9701   
...                                         ...                    ...   
random_forest 2026-06-09                 0.9933                 0.9715   
              2026-06-16                 0.9945                 1.0077   
              2026-06-23                 1.0361                 1.0583   
              2026-06-30                 0.9929                 0.9864   
              2026-07-07                 0.9939                 0.9791   

                          p_top8_no_performance  p_top10_no_performance  
model         date                                                       
decay         2025-11-18                 0.9609                  0.9713  
              2025-11-25                 0.9580                  1.0109  
              2025-12-02                 1.0880                  1.1407  
              2025-12-09                 1.0210                  1.0198  
              2025-12-16                 0.8714                  0.8889  
...                                         ...                     ...  
random_forest 2026-06-09                 1.0346                  1.0363  
              2026-06-16                 1.0204                  1.0637  
              2026-06-23                 1.1117                  1.0593  
              2026-06-30                 0.9401                  0.8946  
              2026-07-07                 0.9385                  0.9738  

[136 rows x 8 columns]

## 3. ROI by Player (Kalshi-relevant players)

In [ ]:
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 300)

def player_roi(model):
    df = dfs[model][dfs[model]['username'].isin(USERNAME_TO_PLAYER)].copy()
    df['player'] = df['username'].map(USERNAME_TO_PLAYER)
    return df.groupby('player')[ROI_COLS].agg(['mean', 'count']).round(4)

by_player = {m: player_roi(m) for m in MODELS}

yes8 = pd.DataFrame({
    m: by_player[m][('p_top8_yes_performance', 'mean')]
    for m in MODELS
}).round(4)
yes8.index.name = 'player'
yes8.sort_values('decay', ascending=False)

## 4. ROI Over Time

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)

for ax, k in zip(axes.flat, N_VALUES):
    for m in MODELS:
        ts = dfs[m].groupby('date')[f'p_top{k}_yes_performance'].mean()
        ax.plot(ts.index, ts.values, label=m, color=COLORS[m], alpha=0.8, marker='o', markersize=3)
    ax.axhline(1, color='red', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_yscale('log')
    ax.set_title(f'YES ROI Over Time — Top {k}', fontweight='bold')
    ax.set_ylabel('Mean ROI (log scale)')
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.4)

fig.suptitle('YES Bet ROI by Model (1.0 = breakeven)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Model Comparison Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, k in zip(axes, [3, 8, 10]):
    vals = [dfs[m][f'p_top{k}_yes_performance'].mean() for m in MODELS]
    bars = ax.bar(MODELS, vals, color=[COLORS[m] for m in MODELS], edgecolor='white')
    ax.axhline(1, color='red', linestyle='--', linewidth=1.5, alpha=0.8)
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
    ax.set_title(f'YES ROI — Top {k}', fontweight='bold')
    ax.set_ylabel('Mean ROI')
    ax.tick_params(axis='x', rotation=15)
    ax.set_ylim(0, max(vals) * 1.2)

fig.suptitle('Average YES Bet ROI by Attendance Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Best / Worst Players by YES ROI (Top 8)

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker

cmap = plt.get_cmap('plasma')

fig, axes = plt.subplots(len(MODELS), 2, figsize=(14, 5 * len(MODELS)))

for row, m in enumerate(MODELS):
    col  = ('p_top8_yes_performance', 'mean')
    cnt  = ('p_top8_yes_performance', 'count')
    data = by_player[m][by_player[m][cnt] >= 5]
    norm = mcolors.Normalize(vmin=data[col].min(), vmax=data[col].max())

    for col_idx, (subset, label) in enumerate([
        (data.nlargest(20, col),  'Best'),
        (data.nsmallest(20, col), 'Worst'),
    ]):
        ax = axes[row, col_idx]
        sns.barplot(x=col, y=subset.index, data=subset,
                    palette=cmap(norm(subset[col])), ax=ax, orient='h')
        ax.axvline(1, color='red', linestyle='--', linewidth=1.2, alpha=0.7)
        ax.set_xscale('log')
        ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
        ax.set_xlabel('Mean ROI')
        ax.set_ylabel('')
        ax.set_title(f'{label} players — {m} (Top 8 YES ROI)', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Model Disagreement Analysis

The decay model computes attendance probability as a pure EWMA of past participation — it has no visibility into seasonality, short-window attendance trends, player rating, or performance trajectory. The ML models (`gradient_boost`, `random_forest`, `logistic`) see all of those signals via the feature set in `attendance_models.py`.

**Hypothesised blind spots for the decay model:**
- **Burned-out regulars** — players with a long recent streak whose `att_rate_4w` is already declining; the high `decay_score` keeps p_participate elevated while ML models react to the recent dropoff.
- **Returning heroes** — highly-rated players after a multi-week absence; `decay` is crushed by the gap (`log_weeks_since_last`), but ML models can learn that elite players return.
- **Seasonal patterns** — decay has no `month_sin/cos` encoding; players with strong summer/school-year cycles will confuse it.
- **Rating-driven engagement** — a rising `rating_trend` signals growing investment in the game; decay ignores it.
- **Normalisation artefact** — the EWMA denominator is `1/(1 - λ)` (infinite-horizon), so a player who attended every week for a year then stopped 4 weeks ago still has a substantial decay score, even though recent evidence strongly contradicts future attendance.

We build a composite DataFrame that aligns all four models on every `(date, username)` pair, then measure `decay_vs_ml = p_decay − mean(p_ML)`.  If the ML models are better calibrated, rows with `decay_vs_ml > 0` (decay over-estimates) should show **lower** actual attendance than decay predicts, and vice-versa.

In [ ]:
POST_ERA = '2025-09-02'

base = (dfs['decay'][['date', 'username', 'participated', 'p_participate']]
        .query("date >= @POST_ERA")
        .rename(columns={'p_participate': 'p_decay'}))

for m in ['gradient_boost', 'random_forest', 'logistic']:
    tmp = (dfs[m][['date', 'username', 'p_participate']]
           .query("date >= @POST_ERA")
           .rename(columns={'p_participate': f'p_{m}'}))
    base = base.merge(tmp, on=['date', 'username'], how='inner')

P_COLS  = ['p_decay', 'p_gradient_boost', 'p_random_forest', 'p_logistic']
ML_COLS = ['p_gradient_boost', 'p_random_forest', 'p_logistic']

base['ml_mean']     = base[ML_COLS].mean(axis=1)
base['mean_p']      = base[P_COLS].mean(axis=1)
base['std_p']       = base[P_COLS].std(axis=1)
base['range_p']     = base[P_COLS].max(axis=1) - base[P_COLS].min(axis=1)
base['decay_vs_ml'] = base['p_decay'] - base['ml_mean']

disagree_df = base[base['mean_p'] > 0.01].copy().reset_index(drop=True)

print(f"Post-era pool rows (mean_p > 0.01):  {len(disagree_df):,}")
print(f"Actual participation rate:            {disagree_df['participated'].mean():.4f}")
print(f"\nCross-model std   mean={disagree_df['std_p'].mean():.4f}  "
      f"p50={disagree_df['std_p'].median():.4f}  "
      f"p90={disagree_df['std_p'].quantile(0.90):.4f}")
print(f"High-disagreement (std > 0.10):  {(disagree_df['std_p'] > 0.10).sum():,} rows  "
      f"({(disagree_df['std_p'] > 0.10).mean():.1%})")
disagree_df[P_COLS + ['ml_mean', 'std_p', 'decay_vs_ml', 'participated']].describe().round(4)

In [ ]:
THRESH = 0.08   # minimum cross-model std to call it "meaningful disagreement"

buckets = [
    ('decay OVER  (decay > ML, std > {})'.format(THRESH),
     disagree_df[(disagree_df['std_p'] > THRESH) & (disagree_df['decay_vs_ml'] >  0)]),
    ('decay UNDER (decay < ML, std > {})'.format(THRESH),
     disagree_df[(disagree_df['std_p'] > THRESH) & (disagree_df['decay_vs_ml'] <= 0)]),
    ('models AGREE (std ≤ {})'.format(THRESH),
     disagree_df[disagree_df['std_p'] <= THRESH]),
]

header = f"  {'Bucket':<46}  {'n':>7}  {'actual_attend':>14}  {'p_decay':>9}  {'p_ml':>9}  {'decay_err':>11}"
print(header)
print('  ' + '-' * (len(header) - 2))
for name, sub in buckets:
    err = (sub['p_decay'] - sub['participated']).mean()
    print(f"  {name:<46}  {len(sub):>7,}  "
          f"{sub['participated'].mean():>14.4f}  "
          f"{sub['p_decay'].mean():>9.4f}  "
          f"{sub['ml_mean'].mean():>9.4f}  "
          f"{err:>+11.4f}")

  Bucket                                                n   actual_attend    p_decay       p_ml    decay_err
  ----------------------------------------------------------------------------------------------------------
  decay OVER  (decay > ML, std > 0.08)                165          0.2848     0.5380     0.4812      +0.2531
  decay UNDER (decay < ML, std > 0.08)             23,757          0.2596     0.2302     0.4697      -0.0294
  models AGREE (std ≤ 0.08)                        46,806          0.1362     0.1372     0.1624      +0.0010


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
ax.hist(disagree_df['std_p'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(THRESH, color='red', linestyle='--', linewidth=1.2, label=f'threshold={THRESH}')
ax.set_xlabel('Std of p_participate across 4 models')
ax.set_ylabel('Player-tournament pairs')
ax.set_title('Cross-Model Disagreement\nDistribution', fontweight='bold')
ax.legend(fontsize=8)

ax = axes[1]
sc = ax.scatter(
    disagree_df['ml_mean'], disagree_df['p_decay'],
    c=disagree_df['participated'], cmap='RdYlGn',
    alpha=0.12, s=4, vmin=0, vmax=1, rasterized=True,
)
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.5)
ax.set_xlabel('ML consensus  p_participate')
ax.set_ylabel('Decay  p_participate')
ax.set_title('Decay vs ML Consensus\n(green=attended, red=no-show)', fontweight='bold')
plt.colorbar(sc, ax=ax, label='attended', shrink=0.8)

ax = axes[2]
edges  = [-1, -0.20, -0.10, -0.05, 0.05, 0.10, 0.20, 1.0]
labels = ['<−0.20', '−0.20:−0.10', '−0.10:−0.05', '±0.05', '0.05:0.10', '0.10:0.20', '>0.20']
disagree_df['gap_bin'] = pd.cut(disagree_df['decay_vs_ml'], bins=edges, labels=labels)
gb = disagree_df.groupby('gap_bin', observed=True)['participated'].agg(['mean', 'count'])

bar_colors = ['#d62728' if l in ('<−0.20', '−0.20:−0.10', '−0.10:−0.05') else
              '#2ca02c' if l in ('0.05:0.10', '0.10:0.20', '>0.20') else
              'steelblue' for l in gb.index]
ax.bar(range(len(gb)), gb['mean'], color=bar_colors, edgecolor='white', alpha=0.85)
ax.axhline(disagree_df['participated'].mean(), color='red', linestyle='--',
           linewidth=1, alpha=0.6, label='overall rate')
ax.set_xticks(range(len(gb)))
ax.set_xticklabels(gb.index, rotation=35, ha='right', fontsize=8)
ax.set_xlabel('decay − ML consensus  (p_participate)')
ax.set_ylabel('Actual participation rate')
ax.set_title('Does disagreement direction\npredict actual attendance?', fontweight='bold')
ax.legend(fontsize=8)
for i, (_, row) in enumerate(gb.iterrows()):
    ax.text(i, row['mean'] + 0.003, f"n={int(row['count'])}", ha='center', fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
FEATURE_COLS = [
    "month_sin", "month_cos", "week_sin", "week_cos",
    "log_weeks_since_last", "att_rate_4w", "att_rate_8w", "att_rate_16w",
    "decay_score", "log_n_appearances",
    "avg_rank_pct_recent", "avg_rank_pct_all",
    "current_rating", "rating_trend",
]

feat_df = pd.read_parquet('models/dataset_cache_post2025.parquet')
feat_df['date'] = pd.to_datetime(feat_df['date'])

merged_feats = disagree_df.merge(
    feat_df[['username', 'date'] + FEATURE_COLS],
    on=['username', 'date'], how='left',
)

decay_hi_f = merged_feats[(merged_feats['std_p'] > THRESH) & (merged_feats['decay_vs_ml'] >  0)]
decay_lo_f = merged_feats[(merged_feats['std_p'] > THRESH) & (merged_feats['decay_vs_ml'] <= 0)]
agree_f    = merged_feats[merged_feats['std_p'] <= THRESH]

col_over  = f'decay_OVER  (n={len(decay_hi_f):,})'
col_under = f'decay_UNDER (n={len(decay_lo_f):,})'
col_agree = f'models_AGREE (n={len(agree_f):,})'

seg = pd.DataFrame({
    col_over:  decay_hi_f[FEATURE_COLS].mean(),
    col_under: decay_lo_f[FEATURE_COLS].mean(),
    col_agree: agree_f[FEATURE_COLS].mean(),
}).round(4)

seg['over_Δ_vs_agree']  = (seg[col_over]  - seg[col_agree]).round(4)
seg['under_Δ_vs_agree'] = (seg[col_under] - seg[col_agree]).round(4)

print("Feature means by disagreement bucket (sorted by |over_Δ|, i.e. what drives decay to be HIGH vs ML)")
seg.sort_values('over_Δ_vs_agree', key=abs, ascending=False)

Feature means by disagreement bucket (sorted by |over_Δ|, i.e. what drives decay to be HIGH vs ML)


,decay_OVER (n=165),"decay_UNDER (n=23,757)","models_AGREE (n=46,806)",over_Δ_vs_agree,under_Δ_vs_agree
current_rating,2762.1316,2696.7754,2649.1025,113.0291,47.6729
log_n_appearances,2.5360,2.1819,1.7159,0.8201,0.4660
rating_trend,4.7434,9.1772,3.9858,0.7576,5.1914
log_weeks_since_last,1.2282,1.2114,1.5908,-0.3626,-0.3794
att_rate_16w,0.6723,0.3222,0.3214,0.3509,0.0008
att_rate_8w,0.6480,0.3642,0.3118,0.3362,0.0524
month_cos,0.7225,-0.2642,0.3901,0.3324,-0.6543
month_sin,0.1894,0.4885,0.4648,-0.2754,0.0237
week_cos,0.7106,-0.1976,0.4435,0.2671,-0.6411
decay_score,0.5176,0.3071,0.2615,0.2561,0.0456


In [ ]:
top_dis = (
    disagree_df[['date', 'username', 'participated'] + P_COLS + ['ml_mean', 'std_p', 'decay_vs_ml']][disagree_df['username'].isin(USERNAME_TO_PLAYER)]
    .nlargest(25, 'std_p')
    .reset_index(drop=True)
    .round(4)
)
print("Top 25 most-disputed player-tournament pairs:")
top_dis

Top 25 most-disputed player-tournament pairs:


,date,username,participated,p_decay,p_gradient_boost,p_random_forest,p_logistic,ml_mean,std_p,decay_vs_ml
0,2026-07-07,penguingm1,0,0.0267,0.8723,0.7193,0.2760,0.6225,0.3908,-0.5958
1,2026-07-07,DanielNaroditsky,0,0.0017,0.8185,0.6538,0.1855,0.5526,0.3844,-0.5509
2,2026-07-07,RaunakSadhwani2005,0,0.0233,0.8222,0.6764,0.1796,0.5594,0.3841,-0.5361
3,2026-07-07,AlmasRakhmatullaev,1,0.0142,0.8207,0.6746,0.2085,0.5679,0.3805,-0.5537
4,2026-07-07,Salem-AR,0,0.0085,0.8214,0.6836,0.2492,0.5847,0.3774,-0.5762
5,2026-07-07,VincentKeymer,0,0.0427,0.8727,0.6708,0.3444,0.6293,0.3652,-0.5866
6,2026-07-07,vugarrasulov,0,0.0547,0.8542,0.6810,0.2896,0.6083,0.3638,-0.5536
7,2026-07-07,LyonBeast,0,0.0853,0.8686,0.7033,0.3825,0.6515,0.3477,-0.5662
8,2026-07-07,Njal28,0,0.0926,0.8434,0.6892,0.3323,0.6216,0.3403,-0.5290
9,2026-07-07,ChessWarrior7197,0,0.1224,0.8888,0.7430,0.4465,0.6928,0.3394,-0.5704


## 8. Participation Calibration & Floor Analysis

When the decay model assigns very low probabilities, strong players returning from a long hiatus still show up at a non-trivial rate — actual p(participate) "sticks" around 2% even when the model predicts < 1%.  This suggests adding a probability floor for all non-cut players.

**Note:** the ML models already have an implicit floor at ~1% (their `predict_proba` output never goes below `0.0101` in the backtest data).  The calibration chart shows where each model's curve **departs from the y = x diagonal** — the flat region at the low end is where a floor would activate.

`p = 0` rows (cut / no-history players) are excluded from all bins; those should remain at `p = 0` regardless of any floor.

In [ ]:
MIN_N = 50

bin_edges = np.unique(np.concatenate([
    np.logspace(-4, -1, 30),
    np.linspace(0.10, 0.90, 17),
    np.array([0.92, 0.94, 0.96, 0.98, 0.99, 0.995, 1.001]),
]))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

panels = [
    (0.001, 0.15, 0.005, 0.30,  True,  'Low-probability region (0.1% – 15%)'),
    (0.001, 1.00, 0.005, 1.10,  False, 'Full range'),
]

FLOOR_CANDIDATES = [
    (0.010, ':',  '1.0%'),
    (0.015, '-.', '1.5%'),
    (0.020, '--', '2.0%'),
    (0.025, ':',  '2.5%'),
]

for ax, (xlim_lo, xlim_hi, ylim_lo, ylim_hi, show_floor, title) in zip(axes, panels):
    for m in MODELS:
        df_m = dfs[m][dfs[m]['p_participate'] > 0].copy()
        df_m['p_bin'] = pd.cut(df_m['p_participate'], bins=bin_edges)
        cal = (df_m.groupby('p_bin', observed=True)['participated']
               .agg(['mean', 'count'])
               .query('count >= @MIN_N and mean > 0'))
        mids = [np.sqrt(b.left * b.right) for b in cal.index]
        ax.plot(mids, cal['mean'], label=m, color=COLORS[m], linewidth=2, zorder=3)

    xs = np.logspace(np.log10(xlim_lo), np.log10(xlim_hi), 200)
    ax.plot(xs, xs, 'k--', linewidth=1, alpha=0.4, label='y = x (perfect)')

    if show_floor:
        for fval, ls, flabel in FLOOR_CANDIDATES:
            ax.axhline(fval, color='#999', linestyle=ls, linewidth=1.2, alpha=0.7,
                       label=f'floor {flabel}')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlim(xlim_lo, xlim_hi)
    ax.set_ylim(ylim_lo, ylim_hi)
    ax.set_xlabel('Predicted p_participate (log scale)')
    ax.set_ylabel('Actual participation rate (log scale)')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3, which='both')

fig.suptitle(
    'Calibration: predicted vs actual p(participate)\n'
    'p = 0 rows excluded — cut / no-history players should always stay at p = 0',
    fontsize=12, fontweight='bold',
)
plt.tight_layout()
plt.show()

In [ ]:
bins_1mp = np.unique(np.concatenate([
    np.logspace(-4, -1, 40),
    np.linspace(0.10, 1.00, 10),
]))

fig, ax = plt.subplots(figsize=(8, 6))

for m in MODELS:
    df_m = dfs[m][dfs[m]['p_participate'] > 0].copy()
    df_m['one_minus_p'] = 1.0 - df_m['p_participate']
    df_m['no_show']     = 1   - df_m['participated']
    df_m['p_bin'] = pd.cut(df_m['one_minus_p'], bins=bins_1mp)
    cal = (df_m.groupby('p_bin', observed=True)['no_show']
           .agg(['mean', 'count'])
           .query('count >= @MIN_N and mean > 0'))
    mids = [np.sqrt(b.left * b.right) for b in cal.index]
    ax.plot(mids, cal['mean'], label=m, color=COLORS[m], linewidth=2)

xs = np.logspace(-4, 0, 300)
ax.plot(xs, xs, 'k--', linewidth=1, alpha=0.4, label='y = x (perfect)')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(0.001, 0.20)
ax.set_ylim(0.001, 1.00)
ax.set_xlabel('Predicted  1 − p_participate  (log scale)')
ax.set_ylabel('Actual no-show rate  (log scale)')
ax.set_title(
    'High-confidence calibration: predicted vs actual p(no-show)\n'
    'Zoomed to 1 − p < 20%  (i.e. model predicts > 80% attendance)',
    fontweight='bold',
)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(8, 24))

for k, (model, model_df) in enumerate(dfs.items()):
    p_participate = []
    avg_participate = []
    n = []
    tail=0.0

    for head in np.geomspace(0.0001,1):
        bin = model_df[(model_df['p_participate']>tail) & (model_df['p_participate']<head)]
        p_participate.append(bin['p_participate'].mean())
        avg_participate.append(bin['participated'].mean())
        n.append(bin['participated'].count())

        tail=head

    sns.scatterplot(x=p_participate, y=avg_participate, ax=axes[k], size=n, hue=(np.array(n)))

    axes[k].plot(p_participate, p_participate, 'k--', linewidth=1, alpha=0.4, label='y = x (perfect calibration)')

    axes[k].set_xlabel('Predicted  p(participate) > X')
    axes[k].set_xscale('log')
    axes[k].set_yscale('log')
    axes[k].set_ylabel('Mean p(participate)')
    axes[k].set_title(
        f'Predicted vs. Actual Participation Rate, {model}\nLow-end, log scale.',
        fontweight='bold',
    )
    axes[k].legend(fontsize=9)
    axes[k].grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(8, 24))

for k, (model, model_df) in enumerate(dfs.items()):
    p_participate = []
    avg_participate = []
    n = []

    for i in np.linspace(0,1):
        bin = model_df[(model_df['p_participate']>i) & (model_df['p_participate']<i+.01)]
        p_participate.append(bin['p_participate'].mean())
        avg_participate.append(bin['participated'].mean())
        n.append(bin['participated'].count())

    sns.scatterplot(x=p_participate, y=avg_participate, ax=axes[k], size=n, hue=(np.array(n) > 100))

    axes[k].plot(p_participate, p_participate, 'k--', linewidth=1, alpha=0.4, label='y = x (perfect calibration)')

    axes[k].set_xlabel('Predicted  p(participate) > X')
    axes[k].set_ylabel('Mean p(participate)')
    axes[k].set_title(
        f'Predicted vs. Actual Participation Rate, {model}\nLow-end.',
        fontweight='bold',
    )
    axes[k].legend(fontsize=9)
    axes[k].grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

In [89]:
import numpy as np
import joblib
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split

decay_df = dfs['decay']

# --- 1. Prepare your data ---
p_participate = np.array(decay_df['p_participate'].values)
participated = np.array(decay_df['participated'].values)

iso_reg = joblib.load('models/isotonic_regression.joblib')

iso_floor = np.where((p_participate > .05),p_participate, iso_reg.predict(p_participate))

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(18,8))

mean_preds=[]
mean_true=[]
pops=[]
tail= 0.0
for head in np.geomspace(0.0001,1):
    indices = (iso_floor>tail)&(iso_floor<head)
    mean_preds.append(np.mean(iso_floor[indices]))
    mean_true.append(np.mean(participated[indices]))
    pops.append(indices.sum())
    tail=head

sns.scatterplot(x=mean_preds,y=mean_true, hue=(np.array(pops)>100),ax=ax[0])
ax[0].plot(mean_preds,mean_preds, linestyle='--', linewidth=1, alpha=0.4, label='y = x (perfect calibration)')

ax[0].set_xlabel('Pred p(participate)')
ax[0].set_xscale('log')
ax[0].set_yscale('log')
ax[0].set_ylabel('True p(participate)')
ax[0].set_title(
    f'Predicted vs. Actual Participation Rate\nIsotropic Regression Model (Decay base model). Low-end, log scale.',
    fontweight='bold',
)
ax[0].legend(fontsize=9)
ax[0].grid(True, alpha=0.3, which='both')

mean_preds=[]
mean_true=[]
pops=[]

for i in np.linspace(0,1):
    indices = (iso_floor>i)&(iso_floor<i+.01)
    mean_preds.append(np.mean(iso_floor[indices]))
    mean_true.append(np.mean(participated[indices]))
    pops.append(indices.sum())

sns.scatterplot(x=mean_preds,y=mean_true,hue=(np.array(pops)>100), ax=ax[1])
ax[1].plot(mean_preds,mean_preds, linestyle='--', linewidth=1, alpha=0.4, label='y = x (perfect calibration)')

ax[1].set_xlabel('Pred p(participate)')
ax[1].set_ylabel('True p(participate)')
ax[1].set_title(
    f'Predicted vs. Actual Participation Rate\nIsotropic Regression Model (Decay base model). Full-width.',
    fontweight='bold',
)
ax[1].legend(fontsize=9)
ax[1].grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()